## **Data Loading & Exploration**

In this section we load the Lending Club dataset and perform
initial exploration to understand the structure and target variable distribution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('/content/loan.csv')

/tmp/ipython-input-2940402872.py:1: DtypeWarning: Columns (19,47,55,112,123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('/content/loan.csv')


In [ ]:
df.shape

(2260668, 145)

In [ ]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Columns: 145 entries, id to settlement_term
dtypes: float64(105), int64(4), object(36)
memory usage: 2.4+ GB


In [ ]:
df.isnull().sum()

,0
id,2260668
member_id,2260668
loan_amnt,0
funded_amnt,0
funded_amnt_inv,0
...,...
settlement_status,2227612
settlement_date,2227612
settlement_amount,2227612
settlement_percentage,2227612


In [ ]:
df['loan_status'].value_counts()

,count
loan_status,
Fully Paid,1041952
Current,919695
Charged Off,261655
Late (31-120 days),21897
In Grace Period,8952
Late (16-30 days),3737
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761
Default,31


# 2. Data Cleaning & Leakage Removal
We remove post-default leakage columns and administrative columns
that carry no predictive value at loan origination.



In [ ]:
missing_percentages = df.isnull().sum() / len(df) * 100
drop_columns = missing_percentages[missing_percentages > 40].index

print(f"Shape of DataFrame before dropping columns: {df.shape}")
print(f"\nColumns to be dropped (more than 40% missing values):\n{drop_columns.tolist()}")

df_cleaned = df.drop(columns=drop_columns)
print(f"\nShape of DataFrame after dropping columns: {df_cleaned.shape}")

Shape of DataFrame before dropping columns: (2260668, 145)

Columns to be dropped (more than 40% missing values):
['id', 'member_id', 'url', 'desc', 'mths_since_last_delinq', 'mths_since_last_record', 'next_pymnt_d', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'mths_since_rcnt_il', 'il_util', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount'

In [ ]:
display(df_cleaned.head())


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,disbursement_method,debt_settlement_flag
0,2500,2500,2500.0,36 months,13.56,84.92,C,C1,Chef,10+ years,...,0.0,1.0,0.0,60124.0,16901.0,36500.0,18124.0,N,Cash,N
1,30000,30000,30000.0,60 months,18.94,777.23,D,D2,Postmaster,10+ years,...,0.0,1.0,0.0,372872.0,99468.0,15000.0,94072.0,N,Cash,N
2,5000,5000,5000.0,36 months,17.97,180.69,D,D1,Administrative,6 years,...,0.0,0.0,0.0,136927.0,11749.0,13800.0,10000.0,N,Cash,N
3,4000,4000,4000.0,36 months,18.94,146.51,D,D2,IT Supervisor,10+ years,...,100.0,0.0,0.0,385183.0,36151.0,5000.0,44984.0,N,Cash,N
4,30000,30000,30000.0,60 months,16.14,731.78,C,C4,Mechanic,10+ years,...,0.0,0.0,0.0,157548.0,29674.0,9300.0,32332.0,N,Cash,N


In [ ]:
potential_leakage_columns = [
    'out_prncp', # Outstanding principal: non-zero values imply the loan is ongoing or not fully paid.
    'out_prncp_inv', # Outstanding principal for investor portion.
    'total_pymnt', # Total payments received to date. High values are correlated with 'Fully Paid'.
    'total_pymnt_inv', # Total payments received from investor portion.
    'total_rec_prncp', # Total principal received.
    'total_rec_int', # Total interest received.
    'total_rec_late_fee', # Total late fees received. Indicates late payments.
    'recoveries', # Post charge-off collection amounts. Directly indicates a 'Charged Off' status.
    'collection_recovery_fee', # Fee for collection efforts. Also indicates severe delinquency.
    'last_pymnt_d', # Date of last payment. Indicates activity/status.
    'last_pymnt_amnt', # Amount of last payment. Can indicate full payment.
    'next_pymnt_d', # Date of next payment. Often null if loan is fully paid or charged off.
    'last_credit_pull_d', # Date of last credit pull. Updated frequently, but post-loan information.
    'debt_settlement_flag' # Indicates if the debt was settled. This happens after issues arise.
]

# Filter to include only columns actually present in df_cleaned
leakage_columns_present = [col for col in potential_leakage_columns if col in df_cleaned.columns]

print("Potential Data Leakage Columns:")
for col in leakage_columns_present:
    print(f"- {col}")

print("\nThese columns provide information that would not be available at the time a loan application is being evaluated for approval, as they reflect the loan's performance or outcome *after* it has been issued.")

Potential Data Leakage Columns:
- out_prncp
- out_prncp_inv
- total_pymnt
- total_pymnt_inv
- total_rec_prncp
- total_rec_int
- total_rec_late_fee
- recoveries
- collection_recovery_fee
- last_pymnt_d
- last_pymnt_amnt
- last_credit_pull_d
- debt_settlement_flag

These columns provide information that would not be available at the time a loan application is being evaluated for approval, as they reflect the loan's performance or outcome *after* it has been issued.


In [ ]:
df_newclean = df_cleaned.drop(columns=leakage_columns_present)
df_newclean2=df_newclean.drop(columns=['chargeoff_within_12_mths','hardship_flag','pymnt_plan','policy_code','title','zip_code','funded_amnt','funded_amnt_inv'])

In [ ]:
df_newclean2.shape

(2260668, 78)

In [ ]:
df_newclean2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 78 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   loan_amnt                   int64  
 1   term                        object 
 2   int_rate                    float64
 3   installment                 float64
 4   grade                       object 
 5   sub_grade                   object 
 6   emp_title                   object 
 7   emp_length                  object 
 8   home_ownership              object 
 9   annual_inc                  float64
 10  verification_status         object 
 11  issue_d                     object 
 12  loan_status                 object 
 13  purpose                     object 
 14  addr_state                  object 
 15  dti                         float64
 16  delinq_2yrs                 float64
 17  earliest_cr_line            object 
 18  inq_last_6mths              float64
 19  open_acc             

In [ ]:
# Identify numeric columns
numeric_cols = df_newclean2.select_dtypes(include=np.number).columns

# Fill missing values in numeric columns with the mean
for col in numeric_cols:
    if df_newclean2[col].isnull().any():
        mean_value = df_newclean2[col].mean()
        df_newclean2[col].fillna(mean_value, inplace=True)
        print(f"Filled missing values in '{col}' with mean: {mean_value:.2f}")

print("\nMissing values after imputation:")
display(df_newclean2.isnull().sum()[df_newclean2.isnull().sum() > 0])

/tmp/ipython-input-2493971293.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_newclean2[col].fillna(mean_value, inplace=True)


Filled missing values in 'annual_inc' with mean: 77992.43
Filled missing values in 'dti' with mean: 18.82
Filled missing values in 'delinq_2yrs' with mean: 0.31
Filled missing values in 'inq_last_6mths' with mean: 0.58
Filled missing values in 'open_acc' with mean: 11.61
Filled missing values in 'pub_rec' with mean: 0.20
Filled missing values in 'revol_util' with mean: 50.34
Filled missing values in 'total_acc' with mean: 24.16
Filled missing values in 'collections_12_mths_ex_med' with mean: 0.02
Filled missing values in 'acc_now_delinq' with mean: 0.00
Filled missing values in 'tot_coll_amt' with mean: 232.73
Filled missing values in 'tot_cur_bal' with mean: 142492.20
Filled missing values in 'open_acc_6m' with mean: 0.93
Filled missing values in 'open_act_il' with mean: 2.78
Filled missing values in 'open_il_12m' with mean: 0.68
Filled missing values in 'open_il_24m' with mean: 1.56
Filled missing values in 'total_bal_il' with mean: 35506.65
Filled missing values in 'open_rv_12m' wit

,0
emp_title,166969
emp_length,146907
earliest_cr_line,29


In [ ]:
df_newclean2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 78 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   loan_amnt                   int64  
 1   term                        object 
 2   int_rate                    float64
 3   installment                 float64
 4   grade                       object 
 5   sub_grade                   object 
 6   emp_title                   object 
 7   emp_length                  object 
 8   home_ownership              object 
 9   annual_inc                  float64
 10  verification_status         object 
 11  issue_d                     object 
 12  loan_status                 object 
 13  purpose                     object 
 14  addr_state                  object 
 15  dti                         float64
 16  delinq_2yrs                 float64
 17  earliest_cr_line            object 
 18  inq_last_6mths              float64
 19  open_acc             

In [ ]:
# Identify object columns with missing values
object_cols_with_nulls = df_newclean2.select_dtypes(include='object').isnull().sum()
object_cols_with_nulls = object_cols_with_nulls[object_cols_with_nulls > 0].index

print("Object columns with missing values:")
print(object_cols_with_nulls.tolist())

# Impute missing values in object columns with the mode
for col in object_cols_with_nulls:
    mode_value = df_newclean2[col].mode()[0] # .mode() can return multiple values, so take the first
    df_newclean2[col].fillna(mode_value, inplace=True)
    print(f"Filled missing values in '{col}' with mode: '{mode_value}'")

print("\nMissing values after categorical imputation:")
display(df_newclean2.isnull().sum()[df_newclean2.isnull().sum() > 0])

Object columns with missing values:
['emp_title', 'emp_length', 'earliest_cr_line']
Filled missing values in 'emp_title' with mode: 'Teacher'


/tmp/ipython-input-2719939072.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_newclean2[col].fillna(mode_value, inplace=True)


Filled missing values in 'emp_length' with mode: '10+ years'
Filled missing values in 'earliest_cr_line' with mode: 'Sep-2004'

Missing values after categorical imputation:


,0


In [ ]:
df_newclean2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 78 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   loan_amnt                   int64  
 1   term                        object 
 2   int_rate                    float64
 3   installment                 float64
 4   grade                       object 
 5   sub_grade                   object 
 6   emp_title                   object 
 7   emp_length                  object 
 8   home_ownership              object 
 9   annual_inc                  float64
 10  verification_status         object 
 11  issue_d                     object 
 12  loan_status                 object 
 13  purpose                     object 
 14  addr_state                  object 
 15  dti                         float64
 16  delinq_2yrs                 float64
 17  earliest_cr_line            object 
 18  inq_last_6mths              float64
 19  open_acc             

In [ ]:
df_filtered = df_newclean2[df_newclean2['loan_status'].isin(['Fully Paid', 'Charged Off'])]

print(f"Shape of DataFrame before filtering: {df_newclean2.shape}")
print(f"Shape of DataFrame after filtering: {df_filtered.shape}")

display(df_filtered['loan_status'].value_counts())

Shape of DataFrame before filtering: (2260668, 78)
Shape of DataFrame after filtering: (1303607, 78)


,count
loan_status,
Fully Paid,1041952
Charged Off,261655


In [ ]:
df_newclean2 = df_filtered.copy()

In [ ]:
df_newclean2.columns

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'issue_d', 'loan_status', 'purpose',
       'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'initial_list_status', 'collections_12_mths_ex_med',
       'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal',
       'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m',
       'total_bal_il', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util',
       'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op',
       'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc',
       'mths_since_recent_bc', 'mths_since_recent_inq',
       'num_accts_ever_120_p

In [ ]:
df_newclean2['sub_grade'].value_counts()

,count
sub_grade,
C1,82815
B4,80606
B5,79580
B3,79424
C2,76854
C3,72731
C4,72108
B2,71689
B1,68853


In [ ]:
# df_newclean2['issue_d'] = pd.to_datetime(df_newclean2['issue_d'], format='%b-%Y')
# df_newclean2['earliest_cr_line'] = pd.to_datetime(df_newclean2['earliest_cr_line'], format='%b-%Y')

# df_newclean2['credit_age_years'] = (df_newclean2['issue_d'] - df_newclean2['earliest_cr_line']).dt.days / 365.25


# df_newclean2['loan_to_income_ratio'] = df_newclean2['loan_amnt'] / df_newclean2['annual_inc']
# df_newclean2['monthly_payment_to_income'] =df_newclean2['installment'] / (df_newclean2['annual_inc'])/12

# df_newclean2['monthly_payment_to_income']

# df_newclean2['high_utilization_flag']=(df_newclean2['revol_util']>75).astype(int)
# df_newclean2['has_publi_rec']=(df_newclean2['pub_rec']>0).astype(int)


# df = pd.get_dummies(df, columns=['purpose', 'home_ownership', 'verification_status',
#                                   'application_type', 'initial_list_status',
#                                   'disbursement_method'], drop_first=True)

In [ ]:

df_newclean2[''].value_counts()

KeyError: ''

In [ ]:
df_newclean2.to_csv('cleaned_loan_data.csv', index=False)
print("DataFrame saved to 'cleaned_loan_data.csv'")

DataFrame saved to 'cleaned_loan_data.csv'
